In [ ]:
from langgraph.graph import StateGraph ,START,END
from langchain_huggingface import HuggingFaceEndpoint , ChatHuggingFace
from typing import TypedDict , Annotated
from dotenv import load_dotenv
from pydantic import Field , BaseModel
import operator
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.exceptions import OutputParserException
from langchain_core.messages import BaseMessage
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import add_messages
from langgraph.checkpoint.memory import InMemorySaver

In [19]:
load_dotenv()  # Load environment variables from .env file
checkpointer=InMemorySaver()
llm = HuggingFaceEndpoint(
repo_id="Qwen/Qwen2.5-7B-Instruct",
task="text-generation",
provider="featherless-ai",
max_new_tokens=1024
) # type: ignore
model = ChatHuggingFace(llm=llm)


In [20]:
class chatbotState(TypedDict):
    message: Annotated[list[BaseMessage], add_messages]


In [24]:
def generate_reply(state: chatbotState) -> dict:
    response = model.invoke(state["message"])
    return {"message": [AIMessage(content=response.content)]}


In [25]:
graph=StateGraph(chatbotState)
thread_id=1
config={'configurable':{'thread_id':thread_id}}

graph.add_node("chat",generate_reply)

graph.add_edge(START,"chat")
graph.add_edge("chat",END)
workflow=graph.compile(checkpointer=checkpointer)

In [26]:

while True:
    user_input = input("Enter input: ").strip()

    if user_input.lower() in ["exit", "stop"]:
        print("Exiting chat.")
        break

    state = {"message": [HumanMessage(content=user_input)]}
    result = workflow.invoke(state, config=config)

    messages = result["message"]
    if isinstance(messages, list) and messages:
        print(messages[-1].content)


The current President of India is Droupadi Murmu. She was sworn in on July 25, 2022, for a five-year term. If you have any specific questions about the President of India or the role, feel free to ask!
We are currently discussing the President of India. India is a federal republic in South Asia. If you have any specific questions about India or its leadership, I'd be happy to help!
The current President of Pakistan is Arif Alvi. He was sworn in on August 19, 2019, for a five-year term.

If you have any specific questions about the President of Pakistan or Pakistan in general, feel free to ask!
We are currently discussing the President of Pakistan. If you have any specific questions about Pakistan or its leadership, feel free to ask!
The first country we talked about was India. We discussed the President of India, Droupadi Murmu.

If you have any specific questions about India or Pakistan, or if you'd like more information on either country, feel free to ask!
Exiting chat.


In [27]:
list(workflow.get_state_history(config))

[StateSnapshot(values={'message': [HumanMessage(content='president of india', additional_kwargs={}, response_metadata={}, id='e6af6f2b-2302-4518-870e-fb66c4a8c930'), HumanMessage(content='preisdent of india', additional_kwargs={}, response_metadata={}, id='244c9a7c-ce31-4cfb-8045-5fdeab0e747e'), AIMessage(content='The current President of India is Droupadi Murmu. She was sworn in on July 25, 2022, for a five-year term. If you have any specific questions about the President of India or the role, feel free to ask!', additional_kwargs={}, response_metadata={}, id='3971f0ce-f1b5-4d6e-9370-a88c6648ad62', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='which country are we talking about', additional_kwargs={}, response_metadata={}, id='f35ad2e5-dfcf-456f-80c6-5dabda3952c3'), AIMessage(content="We are currently discussing the President of India. India is a federal republic in South Asia. If you have any specific questions about India or its leadership, I'd be happy to help!", add